# 03 - Conformité RGPD & Data Cleaning

## 1. Identification de la donnée personnelle

In [1]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import pandas as pd
from extract import read_source_csv, rename_columns_for_staging
from clean import (convert_types, remove_duplicates, handle_missing_values,
    handle_invalid_values, add_derived_variables,)
from pseudonymize import pseudonymize_name, pseudonymize_column

df = read_source_csv('../data/raw/Sample - Superstore.csv')
df = rename_columns_for_staging(df)
df.shape

(10064, 21)

## 2. Pseudonymisation (RGPD)

In [ ]:
exemple = df['customer_name'].iloc[0]
print('Avant :', exemple)
print('Après :', pseudonymize_name(exemple))

## 3. Conversion des types (dates, nombres)

In [2]:
df_typed = convert_types(df)
df_typed.dtypes

row_id                    Int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
country                  object
city                     object
state                    object
postal_code               Int64
region                   object
product_id               object
category                 object
sub_category             object
product_name             object
sales                   float64
quantity                  Int64
discount                float64
profit                  float64
dtype: object

## 4. Suppression des doublons

In [3]:
df_dupli = remove_duplicates(df_typed)

[clean] 70 doublon(s) supprimé(s) (sur row_id)


## 5. Traitement des valeurs manquantes

In [4]:
df_no_manq = handle_missing_values(df_dupli)

[clean] 298 ligne(s) supprimée(s) pour clés/dates manquantes obligatoires


## 6. Traitement des valeurs invalides (règles de gestion)

In [5]:
df_valid = handle_invalid_values(df_no_manq)

[clean] 25 ligne(s) invalide(s) supprimée(s) (discount>100% / quantity<0 / ship_date<order_date)
[clean] total lignes supprimées à cette étape : 216


## 7. Variables métier dérivées

In [6]:
print("Nombre de sales manquants avant :", df_valid["sales"].isna().sum())
#
df_final = add_derived_variables(df_valid)
df_final[['order_date', 'ship_date', 'deliverytime', 'prix_unitaire', 'sales', 'profit', 'profit_margin']].head()
print("Nombre de sales manquants après :", df_final["sales"].isna().sum())

Nombre de sales manquants avant : 187
Nombre de sales manquants après : 4


## Bilan du nettoyage

In [7]:
print('Lignes avant nettoyage :', len(df))
print('Lignes après nettoyage :', len(df_final))
taux_completude = 1 - df_final.isna().mean().mean()
print(f'Taux de complétude moyen final : {taux_completude:.2%}')

Lignes avant nettoyage : 10064
Lignes après nettoyage : 9480
Taux de complétude moyen final : 99.67%
